# Learn Modern LangChain

This notebook teaches the latest LangChain patterns for building applications with language models. It covers:

- Installation and environment setup
- Chat models and prompt templates
- Chains and composability
- Conversation memory
- Document retrieval and QA
- Agents and tool use
- Advanced app-building ideas

## 1. Install dependencies

Install the core packages for LangChain app building. In a notebook or terminal, run:

```bash
pip install langchain openai python-dotenv faiss-cpu
```

If you use additional tools like SerpAPI or browser search, install them separately.

## 2. Setup environment variables

LangChain uses model providers such as OpenAI. Set your API key in a `.env` file or environment variable.

```python
import os
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
assert OPENAI_API_KEY, 'Set OPENAI_API_KEY in your environment.'
```

## 3. Chat models and prompt templates

Modern LangChain uses chat models and structured chat prompts. This is the foundation for dialog and application workflows.

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage

chat = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=0.7)
response = chat([
    SystemMessage(content='You are a helpful assistant.'),
    HumanMessage(content='Explain what LangChain is in one sentence.'),
])
print(response.content)

## 4. Prompt templates

Use prompt templates to separate text structure from your application data. This increases reuse and reduces prompt duplication.

In [ ]:
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain import LLMChain

system_template = SystemMessagePromptTemplate.from_template(
    'You are a helpful AI assistant who explains developer concepts clearly.'
)
human_template = HumanMessagePromptTemplate.from_template(
    'Write a short explanation for why {topic} matters to developers.'
)

prompt = ChatPromptTemplate.from_messages([system_template, human_template])
chain = LLMChain(llm=chat, prompt=prompt)
result = chain.run({'topic': 'LangChain'})
print(result)

## 5. Chains and composability

Chains let you compose multiple LLM calls into a workflow. Use them to implement multi-step app logic.

In [ ]:
from langchain.chains import LLMChain, SimpleSequentialChain
from langchain.prompts import PromptTemplate

summary_prompt = PromptTemplate(
    input_variables=['text'],
    template='Summarize the following text in one sentence:
{text}',
)
tweet_prompt = PromptTemplate(
    input_variables=['summary'],
    template='Write a short social media post based on this summary:
{summary}',
)

summary_chain = LLMChain(llm=chat, prompt=summary_prompt, output_key='summary')
tweet_chain = LLMChain(llm=chat, prompt=tweet_prompt)

overall_chain = SimpleSequentialChain(chains=[summary_chain, tweet_chain], verbose=True)
output = overall_chain.run('LangChain is a library that helps developers build applications with language models using chains, prompts, memory, and tools.')
print(output)

## 6. Conversation memory

Memory preserves chat state so the application can maintain context across turns.

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

conversation_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template('You are a helpful assistant that remembers what the user said earlier.'),
    HumanMessagePromptTemplate.from_template('{chat_history}
Human: {question}
Assistant:'),
])

conversation_chain = LLMChain(llm=chat, prompt=conversation_prompt, memory=memory, verbose=False)

print(conversation_chain.run({'question': 'What is LangChain?'}))
print(conversation_chain.run({'question': 'Now explain it again as if I were a beginner.'}))

## 7. Document retrieval and vector stores

Build a retriever that searches documents based on semantic embeddings. This is the basis for retrieval-augmented generation (RAG).

In [ ]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

texts = [
    'LangChain is a framework for building applications with large language models.',
    'Retrieval augmentation improves accuracy by grounding answers in documents.',
    'Agents allow LangChain apps to call external tools and APIs.',
]

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
docs = splitter.create_documents(texts)

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={'k': 2})
results = retriever.get_relevant_documents('How does LangChain use documents?')
for doc in results:
    print('---')
    print(doc.page_content)

## 8. Retrieval QA

Use the retriever with an LLM to answer questions from your document collection.

In [ ]:
from langchain.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm=chat,
    chain_type='stuff',
    retriever=retriever,
    return_source_documents=True,
)

question = 'What does LangChain use to answer questions from documents?'
result = qa({'query': question})
print('Answer:')
print(result['result'])
print('
Sources:')
for doc in result['source_documents']:
    print('-', doc.page_content)

## 9. Agents and tools

Agents allow your app to call tools, run code, or fetch external data. This is powerful for building interactive applications.

In [ ]:
from langchain.agents import initialize_agent, Tool
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

def get_current_time(query: str) -> str:
    from datetime import datetime
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

time_tool = Tool(
    name='current_time',
    func=get_current_time,
    description='Returns the current date and time when asked.',
)

streaming_callbacks = CallbackManager([StreamingStdOutCallbackHandler()])
agent = initialize_agent(
    tools=[time_tool],
    llm=ChatOpenAI(model_name='gpt-3.5-turbo', streaming=True, callback_manager=streaming_callbacks),
    agent='zero-shot-react-description',
    verbose=True,
)

print(agent.run('What time is it right now?'))

## 10. Advanced app-building topics

Modern LangChain applications often include:

- Streaming and callback handling for responsive UIs
- Structured prompts for reliable outputs
- Hybrid search combining keyword and vector search
- Tool-enabled agents with external APIs or local functions
- Document loaders for PDFs, HTML, and databases
- Deployment patterns using FastAPI, Streamlit, or Gradio

Explore these as the next step once you have the basics working.

## Next steps

1. Add real document loaders for PDF, website, and database content.
2. Build a retrieval QA app with a web interface.
3. Create an agent that uses external tools like search, calendars, or custom APIs.
4. Experiment with chain-of-thought prompts and model selection.